# AgentCar — train model NLU cho edge (bản tự chứa)

Không cần upload gì. Dữ liệu và mã train đã nhúng sẵn trong notebook này.

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**. Rồi Runtime → Run all.


In [ ]:
import torch, subprocess
g = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True).stdout.strip()
print(g or 'KHONG CO GPU — vao Runtime > Change runtime type > T4 GPU')
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 1. Bung mã và dữ liệu (đã nhúng sẵn)


In [ ]:
import base64, io, zipfile, pathlib
GOI_B64 = (
    "..."
)
zipfile.ZipFile(io.BytesIO(base64.b64decode(GOI_B64))).extractall("/content/work")
for f in sorted(pathlib.Path("/content/work").rglob("*")):
    if f.is_file():
        print("  ", f, f"{f.stat().st_size/1000:.1f} KB")


## 2. Cài thư viện và tải MASSIVE vi-VN


In [ ]:
!pip -q install 'transformers>=4.44' sentencepiece
!wget -q https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz
!tar xzf amazon-massive-dataset-1.0.tar.gz 1.0/data/vi-VN.jsonl
!wc -l 1.0/data/vi-VN.jsonl


## 3. Bước 1 — thân PhoBERT làm quen văn phong ra lệnh tiếng Việt

Dùng nhãn của MASSIVE. Xong thì vứt đầu, chỉ giữ thân.


In [ ]:
!cd /content/work && python3 nlu/train.py --giai-doan massive \
    --massive /content/1.0/data/vi-VN.jsonl --epochs 5 --bs 32 --out /content/runs


## 4. Bước 2 — gắn đầu của mình, học 30 thiết bị VF9


In [ ]:
!cd /content/work && python3 nlu/train.py --giai-doan mien \
    --tu-than /content/runs/massive/than.pt --epochs 6 --bs 32 --out /content/runs


## 5. Bước 3 — chưng cất xuống trò 4 lớp (thứ đóng gói lên xe)


In [ ]:
!cd /content/work && python3 nlu/train.py --giai-doan chung-cat \
    --thay /content/runs/mien/model.pt \
    --massive /content/1.0/data/vi-VN.jsonl --epochs 10 --bs 64 --out /content/runs


## 6. Tải kết quả về máy


In [ ]:
!cd /content/runs && zip -qr /content/ket_qua.zip chung-cat mien/ket_qua.json
!ls -lh /content/ket_qua.zip
from google.colab import files; files.download('/content/ket_qua.zip')
